# DebugZero Training Workflow (OpenEnv-backed)

This notebook trains against the deployed `DebugZero` environment instead of embedding local copies of the seed bank, executor, bug injector, or reward functions.

What this notebook does:
- installs and clones the repo
- connects to your deployed Hugging Face OpenEnv app
- builds GRPO training rows from live environment resets and env-verified buggy states
- computes rewards by stepping `DebugzeroEnv`, so the training signal comes from the real environment


In [ ]:
# Notebook + environment configuration
REPO_URL = "https://github.com/Ray-0906/DebugZero.git"
BRANCH = "main"

# Preferred: deployed Hugging Face Space URL.
# A browser URL like https://huggingface.co/spaces/OWNER/SPACE also works.
REMOTE_OPENENV_URL = "https://the-fool-09-debugzero.hf.space"

USE_UNSLOTH = True
MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
FALLBACK_MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
OUTPUT_DIR = "debugzero_openenv_model"

DATASET_ROUNDS = 4
NUM_GENERATIONS = 4
MAX_STEPS = 200
EVAL_SAMPLES = 6
BUG_FOCUS = None
RUN_TRAINING = True
RUN_BASELINE_EVAL = True


In [ ]:
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path


def pip_install(*packages):
    print("Installing:", " ".join(packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


pip_install("--upgrade", "pip")
pip_install(
    "openenv-core[core]>=0.2.1",
    "datasets>=2.20.0",
    "trl>=0.20.0",
    "transformers>=4.51.0",
    "accelerate>=0.34.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.0",
    "matplotlib>=3.8.0",
    "pandas>=2.0.0",
    "thefuzz[speedup]>=0.22.1",
    "uvicorn[standard]>=0.30.0",
    "requests>=2.31.0",
)

if USE_UNSLOTH:
    try:
        pip_install("unsloth")
    except Exception as exc:
        print("Unsloth install failed; falling back to native TRL.")
        print(exc)

REPO_DIR = Path.cwd() / "DebugZero"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(REPO_DIR)])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repo ready at", REPO_DIR)


In [ ]:
import atexit
import os
import subprocess
import sys
import time
from urllib.parse import urlparse

import requests


def normalize_space_url(url: str) -> str:
    url = (url or "").strip().rstrip("/")
    if not url:
        return ""
    parsed = urlparse(url)
    if parsed.netloc == "huggingface.co" and parsed.path.startswith("/spaces/"):
        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 3:
            owner, space = parts[1], parts[2]
            return f"https://{owner}-{space}.hf.space".lower()
    return url


REMOTE_OPENENV_URL = normalize_space_url(REMOTE_OPENENV_URL)

if REMOTE_OPENENV_URL:
    BASE_URL = REMOTE_OPENENV_URL
    server_process = None
else:
    BASE_URL = "http://127.0.0.1:8000"
    server_process = subprocess.Popen(
        [sys.executable, "-m", "debugZero.server.app", "--host", "127.0.0.1", "--port", "8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(REPO_DIR),
    )
    atexit.register(lambda: server_process and server_process.poll() is None and server_process.terminate())


def wait_for_openenv(base_url: str, timeout_s: int = 120):
    deadline = time.time() + timeout_s
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url}/schema", timeout=5)
            if response.status_code == 200:
                return response.json()
            last_error = f"HTTP {response.status_code}: {response.text[:200]}"
        except Exception as exc:
            last_error = exc
        time.sleep(2)

    if server_process and server_process.stdout:
        print("--- OpenEnv server output ---")
        print(server_process.stdout.read())
    raise RuntimeError(f"OpenEnv did not become ready at {base_url}: {last_error}")


schema = wait_for_openenv(BASE_URL)
print("Connected to OpenEnv:", BASE_URL)
schema


In [ ]:
import re
from contextlib import contextmanager

from datasets import Dataset
from debugZero.client import DebugzeroEnv
from debugZero.models import DebugzeroAction
from training.dual_role_sampler import sample_proposer_prompt, sample_solver_prompt


def observation(result):
    return getattr(result, "observation", result)


def extract_code(text):
    if isinstance(text, list):
        if text and isinstance(text[0], dict):
            text = text[0].get("content", "")
        else:
            text = "\n".join(map(str, text))
    text = str(text or "")
    match = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    return (match.group(1) if match else text).strip()


@contextmanager
def seed_session(seed_index: int):
    with DebugzeroEnv(base_url=BASE_URL).sync() as env:
        reset_obs = None
        for _ in range(seed_index + 1):
            reset_obs = observation(env.reset())
        yield env, reset_obs


def collect_seed_snapshots(max_unique: int = 32):
    snapshots = []
    seen = set()
    with DebugzeroEnv(base_url=BASE_URL).sync() as env:
        for seed_index in range(max_unique):
            reset_obs = observation(env.reset())
            seed_id = reset_obs.metadata.get("seed_id", f"seed-{seed_index}")
            if seed_id in seen:
                break
            seen.add(seed_id)
            snapshots.append(
                {
                    "seed_index": seed_index,
                    "seed_id": seed_id,
                    "clean_code": reset_obs.current_code,
                }
            )
    if not snapshots:
        raise RuntimeError("Failed to collect any seeds from the deployed environment.")
    return snapshots


def candidate_bug_variants(clean_code: str):
    replacements = [
        ("idx != idx2", "idx == idx2"),
        ("distance < threshold", "distance <= threshold"),
        ("range(n + 1)", "range(n)"),
        ("return values[1:-1]", "return values[:-1]"),
        ("<= values[idx + 1]", "< values[idx + 1]"),
        ("if len(text) > 0:", "if len(text) >= 0:"),
        ("if values[idx] > best:", "if values[idx] < best:"),
        ("if value == target:", "if value != target:"),
        ("return values[:-1]", "return values[1:]"),
        ("if value > threshold:", "if value >= threshold:"),
        ("result.append(total)", "result.append(value)"),
        ("return True", "return False"),
        ("return False", "return True"),
    ]
    seen = set()
    for old, new in replacements:
        if old in clean_code:
            candidate = clean_code.replace(old, new, 1)
            if candidate != clean_code and candidate not in seen:
                seen.add(candidate)
                yield candidate


def find_verified_bug(seed_index: int, clean_code: str):
    for candidate in candidate_bug_variants(clean_code):
        with seed_session(seed_index) as (env, _reset_obs):
            result = env.step(DebugzeroAction(role="proposer", code=candidate))
            obs = observation(result)
            if (not obs.tests_passed) and (not obs.syntax_error):
                return {
                    "buggy_code": obs.current_code,
                    "execution_result": obs.execution_result,
                    "reward": float(getattr(result, "reward", 0.0) or 0.0),
                }
    return None


seed_snapshots = collect_seed_snapshots()
print("Collected seeds:", [snap["seed_id"] for snap in seed_snapshots])

with seed_session(0) as (env, reset_obs):
    print("Smoke test seed:", reset_obs.metadata.get("seed_id"))
    smoke_bug = next(candidate_bug_variants(reset_obs.current_code), None)
    if smoke_bug is not None:
        prop_result = env.step(DebugzeroAction(role="proposer", code=smoke_bug))
        prop_obs = observation(prop_result)
        print("Proposer reward:", getattr(prop_result, "reward", None), "tests_passed:", prop_obs.tests_passed)


In [ ]:
def build_openenv_dataset(rounds: int = DATASET_ROUNDS) -> Dataset:
    rows = []
    verified_bug_cache = {}

    for snapshot in seed_snapshots:
        verified_bug_cache[snapshot["seed_index"]] = find_verified_bug(snapshot["seed_index"], snapshot["clean_code"])

    missing_solver = [snap["seed_id"] for snap in seed_snapshots if verified_bug_cache[snap["seed_index"]] is None]
    if missing_solver:
        print("No verified solver bug found for:", missing_solver)

    for round_idx in range(rounds):
        for snapshot in seed_snapshots:
            clean_code = snapshot["clean_code"]
            rows.append(
                {
                    "prompt": sample_proposer_prompt(clean_code, bug_focus=BUG_FOCUS),
                    "role": "proposer",
                    "seed_id": snapshot["seed_id"],
                    "seed_index": snapshot["seed_index"],
                    "clean_code": clean_code,
                    "buggy_code": "",
                    "execution_result": "",
                    "round_idx": round_idx,
                }
            )

            bug_case = verified_bug_cache[snapshot["seed_index"]]
            if bug_case is not None:
                rows.append(
                    {
                        "prompt": sample_solver_prompt(bug_case["buggy_code"], bug_case["execution_result"]),
                        "role": "solver",
                        "seed_id": snapshot["seed_id"],
                        "seed_index": snapshot["seed_index"],
                        "clean_code": clean_code,
                        "buggy_code": bug_case["buggy_code"],
                        "execution_result": bug_case["execution_result"],
                        "round_idx": round_idx,
                    }
                )

    return Dataset.from_list(rows)


train_dataset = build_openenv_dataset(rounds=DATASET_ROUNDS)
print(train_dataset)
print(train_dataset[0]["prompt"][:500])


In [ ]:
def rollout_reward(seed_index: int, role: str, submitted_code: str, buggy_code: str = "") -> float:
    with seed_session(seed_index) as (env, _reset_obs):
        if role == "proposer":
            result = env.step(DebugzeroAction(role="proposer", code=submitted_code))
            return float(getattr(result, "reward", 0.0) or 0.0)

        if role == "solver":
            if not buggy_code:
                return 0.0
            proposer_result = env.step(DebugzeroAction(role="proposer", code=buggy_code))
            proposer_obs = observation(proposer_result)
            if proposer_obs.tests_passed or proposer_obs.syntax_error:
                return 0.0
            result = env.step(DebugzeroAction(role="solver", code=submitted_code))
            return float(getattr(result, "reward", 0.0) or 0.0)

    return 0.0


def _column(kwargs, singular, plural=None):
    if singular in kwargs and kwargs[singular] is not None:
        return kwargs[singular]
    if plural and plural in kwargs and kwargs[plural] is not None:
        return kwargs[plural]
    raise KeyError(f"Reward function missing dataset column '{singular}'. Available keys: {sorted(kwargs.keys())}")


def openenv_reward(*args, **kwargs):
    completions = kwargs.get("completions")
    if completions is None:
        if len(args) >= 2:
            completions = args[1]
        elif len(args) == 1:
            completions = args[0]
        else:
            raise TypeError("Reward function did not receive completions.")

    roles = _column(kwargs, "role", "roles")
    seed_indices = _column(kwargs, "seed_index", "seed_indices")
    buggy_codes = kwargs.get("buggy_code", kwargs.get("buggy_codes", [""] * len(completions)))

    rewards = []
    for completion, role, seed_index, buggy_code in zip(completions, roles, seed_indices, buggy_codes):
        code = extract_code(completion)
        rewards.append(rollout_reward(int(seed_index), role, code, buggy_code))
    return rewards


first_solver = next((row for row in train_dataset if row["role"] == "solver"), None)
if first_solver is not None:
    print(
        "Solver reward sanity:",
        openenv_reward(
            [first_solver["prompt"]],
            [f"""```python
{first_solver['clean_code']}
```"""],
            role=["solver"],
            seed_index=[first_solver["seed_index"]],
            buggy_code=[first_solver["buggy_code"]],
        ),
    )


In [ ]:
import torch

HAS_UNSLOTH = False
if USE_UNSLOTH:
    try:
        from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported
        PatchFastRL("GRPO", FastLanguageModel)
        HAS_UNSLOTH = True
    except Exception as exc:
        print("Using native Transformers/TRL fallback because Unsloth is unavailable:")
        print(exc)
        HAS_UNSLOTH = False

if not HAS_UNSLOTH:
    is_bfloat16_supported = lambda: False

from trl import GRPOConfig, GRPOTrainer

if HAS_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=2048,
        load_in_4bit=True,
        fast_inference=False,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        FALLBACK_MODEL_ID,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
def model_device(model):
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def generate_completion(prompt, max_new_tokens=384):
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device(model))
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


def evaluate_policy(dataset, n=4):
    rows = [dataset[i] for i in range(min(n, len(dataset)))]
    completions = [generate_completion(row["prompt"]) for row in rows]
    rewards = openenv_reward(
        [row["prompt"] for row in rows],
        completions,
        role=[row["role"] for row in rows],
        seed_index=[row["seed_index"] for row in rows],
        buggy_code=[row["buggy_code"] for row in rows],
    )
    return rewards, completions


if RUN_BASELINE_EVAL:
    baseline_rewards, baseline_completions = evaluate_policy(train_dataset, n=EVAL_SAMPLES)
else:
    baseline_rewards, baseline_completions = [], []

print("Baseline rewards:", baseline_rewards)
if baseline_rewards:
    print("Baseline mean:", sum(baseline_rewards) / len(baseline_rewards))


In [ ]:
import inspect


def make_grpo_config(**kwargs):
    supported = inspect.signature(GRPOConfig).parameters
    filtered = {key: value for key, value in kwargs.items() if key in supported}
    ignored = sorted(set(kwargs) - set(filtered))
    if ignored:
        print("Ignoring unsupported GRPOConfig args for this TRL version:", ignored)
    return GRPOConfig(**filtered)


training_args = make_grpo_config(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=768,
    max_completion_length=256,
    logging_steps=5,
    save_steps=50,
    report_to="none",
    bf16=bool(torch.cuda.is_available() and is_bfloat16_supported()),
    fp16=bool(torch.cuda.is_available() and not is_bfloat16_supported()),
    remove_unused_columns=False,
)

trainer_kwargs = dict(
    model=model,
    reward_funcs=[openenv_reward],
    args=training_args,
    train_dataset=train_dataset,
)

try:
    trainer = GRPOTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = GRPOTrainer(tokenizer=tokenizer, **trainer_kwargs)

if RUN_TRAINING:
    train_result = trainer.train()
    trainer.save_model(OUTPUT_DIR)
else:
    train_result = None
    print("RUN_TRAINING=False, trainer configured but not executed.")


In [ ]:
trained_rewards, trained_completions = evaluate_policy(train_dataset, n=EVAL_SAMPLES)
print("Baseline rewards:", baseline_rewards)
if baseline_rewards:
    print("Baseline mean:", sum(baseline_rewards) / len(baseline_rewards))
print("Trained rewards:", trained_rewards)
if trained_rewards:
    print("Trained mean:", sum(trained_rewards) / len(trained_rewards))


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

os.makedirs("results", exist_ok=True)
history = pd.DataFrame(getattr(trainer.state, "log_history", []))
history.to_csv("results/training_log.csv", index=False)

reward_cols = [col for col in history.columns if "reward" in col.lower()]
loss_cols = [col for col in history.columns if "loss" in col.lower()]

if "step" in history.columns and reward_cols:
    ax = history.plot(x="step", y=reward_cols, marker="o", figsize=(8, 4))
    ax.set_xlabel("training step")
    ax.set_ylabel("reward")
    ax.set_title("DebugZero OpenEnv reward during GRPO")
    plt.tight_layout()
    plt.savefig("results/reward_curve.png", dpi=160)
    plt.show()
else:
    print("No reward columns found in trainer history. Columns:", list(history.columns))

if "step" in history.columns and loss_cols:
    ax = history.plot(x="step", y=loss_cols, marker="o", figsize=(8, 4))
    ax.set_xlabel("training step")
    ax.set_ylabel("loss")
    ax.set_title("DebugZero GRPO loss")
    plt.tight_layout()
    plt.savefig("results/loss_curve.png", dpi=160)
    plt.show()
else:
    print("No loss columns found in trainer history. Columns:", list(history.columns))

comparison = pd.DataFrame(
    {
        "phase": ["baseline", "trained"],
        "mean_reward": [
            sum(baseline_rewards) / len(baseline_rewards) if baseline_rewards else 0.0,
            sum(trained_rewards) / len(trained_rewards) if trained_rewards else 0.0,
        ],
    }
)
ax = comparison.plot.bar(x="phase", y="mean_reward", legend=False, figsize=(5, 4))
ax.set_xlabel("policy")
ax.set_ylabel("mean live OpenEnv reward")
ax.set_title("Before vs after training")
plt.tight_layout()
plt.savefig("results/baseline_vs_trained_reward.png", dpi=160)
plt.show()
comparison


In [ ]:
print("Sample post-train completions:")
for row, completion, reward in zip(train_dataset.select(range(min(4, len(train_dataset)))), trained_completions[:4], trained_rewards[:4]):
    print("=" * 80)
    print("role:", row["role"], "seed:", row["seed_id"], "reward:", reward)
    print(completion[:1200])
